In [1]:
import h5py, numpy as np, math
from collections import Counter
from shapely import equals, intersects, touches, contains
from shapely.geometry import box as b

DATA = "data"

with h5py.File(f"{DATA}/vg150/VG-SGG.h5", "r") as f:
    boxes_all = f['boxes_1024'][:]
    first_b = f['img_to_first_box'][:]
    last_b = f['img_to_last_box'][:]
    first_r = f['img_to_first_rel'][:]
    last_r = f['img_to_last_rel'][:]
    rels_all = f['relationships'][:]
    splits = f['split'][:]

# convert to corners
xc, yc, w, h = boxes_all[:,0], boxes_all[:,1], boxes_all[:,2], boxes_all[:,3]
corners = np.column_stack([
    (xc - w/2).astype(int), (yc - h/2).astype(int),
    (xc + w/2).astype(int), (yc + h/2).astype(int),
])

def topology(h1, h2):
    hb, tb = b(*h1), b(*h2)
    if equals(hb, tb): return 7
    if not intersects(hb, tb): return 1 if touches(hb, tb) else 0
    if contains(hb, tb): return 3
    if contains(tb, hb): return 4
    return 2

names = {0:'disjoint',1:'touching',2:'overlap',3:'contains',4:'contained',7:'equal'}
print(f"shapely ready, {len(corners)} boxes")


shapely ready, 1145398 boxes


In [2]:
# manual vs shapely comparison
def topology_manual(h, t):
    hx1, hy1, hx2, hy2 = h
    tx1, ty1, tx2, ty2 = t
    if hx1==tx1 and hy1==ty1 and hx2==tx2 and hy2==ty2: return 7
    ov = not (hx2<=tx1 or tx2<=hx1 or hy2<=ty1 or ty2<=hy1)
    if not ov:
        touch = (hx2==tx1 or tx2==hx1 or hy2==ty1 or ty2==hy1)
        return 1 if touch else 0
    hc = hx1<=tx1 and hy1<=ty1 and hx2>=tx2 and hy2>=ty2
    tc = tx1<=hx1 and ty1<=hy1 and tx2>=hx2 and ty2>=hy2
    if hc: return 3
    if tc: return 4
    return 2

tests = [
    ((0,0,10,10),(20,20,30,30)),
    ((0,0,10,10),(10,0,20,10)),
    ((0,0,15,15),(10,10,25,25)),
    ((0,0,30,30),(5,5,15,15)),
    ((5,5,15,15),(0,0,30,30)),
    ((0,0,10,10),(0,0,10,10)),
    ((0,0,10,10),(9,9,20,20)),
]
for hb, tb in tests:
    m = topology_manual(hb, tb)
    s = topology(hb, tb)
    print(f"{names[m]:15s} manual={m} shapely={s} {'OK' if m==s else 'DIFF'}")


disjoint        manual=0 shapely=0 OK
touching        manual=1 shapely=2 DIFF
overlap         manual=2 shapely=2 OK
contains        manual=3 shapely=3 OK
contained       manual=4 shapely=4 OK
equal           manual=7 shapely=7 OK
overlap         manual=2 shapely=2 OK


In [3]:
# edge: float rounding
h1 = (0.5, 0, 10.5, 10)
t1 = (10.0, 0, 20.0, 10)
m = topology_manual(tuple(int(v) for v in h1), tuple(int(v) for v in t1))
s = topology(h1, t1)
print(f"manual(int): {names[m]}, shapely(float): {names[s]}")
print("rounding matters at boundaries")


manual(int): touching, shapely(float): overlap
rounding matters at boundaries


In [4]:
# topology frequency on real data (sampled)
cnt = Counter()
sampled = 0
for img_idx in range(0, len(first_r), 50):
    r_start, r_end = first_r[img_idx], last_r[img_idx] + 1
    b_start, b_end = first_b[img_idx], last_b[img_idx] + 1
    if r_end <= r_start or b_end <= b_start: continue
    img_rels = rels_all[r_start:r_end]
    for (sid, oid) in img_rels[:20]:
        si, oi = sid - b_start, oid - b_start
        if si < 0 or oi < 0 or si >= b_end-b_start or oi >= b_end-b_start: continue
        t = topology(corners[b_start+si], corners[b_start+oi])
        cnt[t] += 1
        sampled += 1
print(f"sampled {sampled} pairs:")
for t, c in cnt.most_common():
    print(f"  {names[t]:15s}: {c:5d} ({c/sampled*100:.1f}%)")


sampled 12002 pairs:
  overlap        :  4551 (37.9%)
  contained      :  3629 (30.2%)
  contains       :  2899 (24.2%)
  disjoint       :   921 (7.7%)
  equal          :     2 (0.0%)


In [5]:
# topology for first 5 images
cnt = Counter()
for img_idx in range(5):
    start, end = first_b[img_idx], last_b[img_idx] + 1
    img_boxes = corners[start:end]
    for i in range(len(img_boxes)):
        for j in range(len(img_boxes)):
            if i == j: continue
            cnt[topology(img_boxes[i], img_boxes[j])] += 1
print("first 5 images (all pairs):")
for t, c in cnt.most_common():
    print(f"  {names[t]:15s}: {c}")


first 5 images (all pairs):
  disjoint       : 702
  overlap        : 180
  contains       : 62
  contained      : 62


In [6]:
# angular direction test
def angle_bin(h, t):
    hx, hy = (h[0]+h[2])/2, (h[1]+h[3])/2
    tx, ty = (t[0]+t[2])/2, (t[1]+t[3])/2
    dx, dy = tx-hx, ty-hy
    if dx==0 and dy==0: return 0
    rad = math.atan2(dy, dx)
    if rad < 0: rad += 2*math.pi
    return int((rad + math.pi/8) // (math.pi/4)) % 8

dirs = ["N","NE","E","SE","S","SW","W","NW"]
h = (0,0,10,10)
for name, tx, ty in [("N",5,-10),("E",20,5),("S",5,20),("W",-10,5),("NE",15,-10),("SE",15,20),("SW",-5,20),("NW",-5,-10)]:
    t = (tx,ty,tx+10,ty+10)
    b = angle_bin(h, t)
    print(f"{name}: bin={b} ({dirs[b]})")


N: bin=7 (NW)
E: bin=0 (N)
S: bin=2 (E)
W: bin=3 (SE)
NE: bin=7 (NW)
SE: bin=1 (NE)
SW: bin=2 (E)
NW: bin=5 (SW)
